# Contoh Proses Normalisasi Kata (Slang Word Normalization)
## DuoSentimen - Analisis Sentimen Ulasan Duolingo
**By Ahmad Saifulla**

---

Normalisasi kata adalah proses mengubah kata-kata tidak baku (slang/singkatan) menjadi bentuk baku.
Hal ini penting karena ulasan di Google Play Store banyak menggunakan bahasa informal.

## 1. Load Kamus Normalisasi

In [ ]:
import json
import pandas as pd
import re

# Load kamus normalisasi
with open('normalisasi.json', 'r', encoding='utf-8') as f:
    kamus_normalisasi = json.load(f)

print(f'Total kata dalam kamus normalisasi: {len(kamus_normalisasi)} kata')
print()

## 2. Tampilkan Isi Kamus Normalisasi

In [ ]:
# Tampilkan kamus dalam bentuk tabel
df_kamus = pd.DataFrame(
    list(kamus_normalisasi.items()),
    columns=['Kata Tidak Baku (Slang)', 'Kata Baku']
)
df_kamus.index = df_kamus.index + 1
df_kamus.index.name = 'No'

print('=== KAMUS NORMALISASI KATA ===')
print(f'Total: {len(df_kamus)} pasangan kata\n')
df_kamus

## 3. Kategori Kamus Normalisasi

In [ ]:
# Kelompokkan berdasarkan kata baku
from collections import Counter

kata_baku_counts = Counter(kamus_normalisasi.values())
print('=== Kata Baku dengan Variasi Slang Terbanyak ===')
for kata, jumlah in kata_baku_counts.most_common(15):
    slang_list = [k for k, v in kamus_normalisasi.items() if v == kata]
    print(f'\n"{kata}" ({jumlah} variasi):')
    print(f'  Slang: {", ".join(slang_list)}')

## 4. Fungsi Normalisasi

In [ ]:
def normalisasi(text, kamus):
    """
    Normalisasi kata slang menjadi kata baku.
    
    Proses:
    1. Pecah teks menjadi kata-kata
    2. Cek setiap kata di kamus normalisasi
    3. Jika ada di kamus, ganti dengan kata baku
    4. Jika tidak ada, biarkan kata asli
    5. Gabungkan kembali
    """
    words = text.split()
    normalized = []
    
    for word in words:
        # Cek di kamus (case-insensitive)
        kata_baku = kamus.get(word.lower(), word)
        normalized.append(kata_baku)
    
    return ' '.join(normalized)

print('Fungsi normalisasi berhasil dibuat!')

## 5. Contoh Proses Normalisasi (Before vs After)

In [ ]:
# Contoh kalimat ulasan yang mengandung kata slang
contoh_ulasan = [
    'apk nya bgs bgt tp lemot sma sering crash',
    'gak bsa login udh update tp msh error',
    'mantap bgt app nya keren poll buat bljr bahasa',
    'jelek parah gk bs dipake org hrs uninstall aja',
    'lumayan oke tp krg fitur buat belajar speaking',
    'bnr2 bagus app ini smgt bljr bahasa inggris',
    'gw suka bgt sm app ini gampang bgt pakenya',
    'knp ya apk nya ngelag trs gmn cara fix nya',
    'makasih dev udh bikin app yg bgus bgt',
    'ancur bgt app nya gbs download pelajaran'
]

print('=' * 80)
print('CONTOH PROSES NORMALISASI KATA')
print('=' * 80)

hasil = []
for i, ulasan in enumerate(contoh_ulasan, 1):
    hasil_normalisasi = normalisasi(ulasan, kamus_normalisasi)
    
    # Cari kata yang berubah
    kata_asli = ulasan.split()
    kata_normal = hasil_normalisasi.split()
    perubahan = []
    
    idx_normal = 0
    for kata in kata_asli:
        kata_baku = kamus_normalisasi.get(kata.lower())
        if kata_baku:
            perubahan.append(f'"{kata}" → "{kata_baku}"')
    
    hasil.append({
        'No': i,
        'Sebelum': ulasan,
        'Sesudah': hasil_normalisasi,
        'Kata yang berubah': ', '.join(perubahan) if perubahan else '-'
    })
    
    print(f'\n[{i}]')
    print(f'  SEBELUM : {ulasan}')
    print(f'  SESUDAH : {hasil_normalisasi}')
    if perubahan:
        print(f'  BERUBAH : {" | ".join(perubahan)}')
    print('-' * 80)

In [ ]:
# Tampilkan dalam bentuk tabel
df_hasil = pd.DataFrame(hasil)
df_hasil = df_hasil.set_index('No')
df_hasil

## 6. Pipeline Preprocessing Lengkap (Termasuk Normalisasi)

Posisi normalisasi dalam pipeline preprocessing:

```
Teks Mentah → Case Folding → Cleaning → [NORMALISASI] → Tokenizing → Stopword Removal → Stemming → Teks Bersih
```

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import os

# Inisialisasi stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Load stopwords
if os.path.exists('stopwords.txt'):
    with open('stopwords.txt', 'r', encoding='utf-8') as f:
        stop_words = set([line.strip() for line in f if line.strip()])
else:
    stop_words = set()

print(f'Stemmer: Sastrawi (Nazief-Adriani)')
print(f'Stopwords: {len(stop_words)} kata')
print(f'Kamus Normalisasi: {len(kamus_normalisasi)} kata')

In [ ]:
# Fungsi preprocessing per tahap
def case_folding(text):
    return text.lower()

def cleaning(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenizing(text):
    return text.split()

def hapus_stopwords(tokens):
    return [w for w in tokens if w not in stop_words]

def stem(tokens):
    return [stemmer.stem(w) for w in tokens]

# Contoh proses tahap demi tahap
teks_contoh = 'Apk nya bgs BGT!! tp lemot sma sering crash 😭😭 @duolingo #fix'

print('PROSES PREPROCESSING TAHAP DEMI TAHAP')
print('=' * 70)
print(f'\n0. TEKS ASLI         : {teks_contoh}')

step1 = case_folding(teks_contoh)
print(f'1. CASE FOLDING      : {step1}')

step2 = cleaning(step1)
print(f'2. CLEANING          : {step2}')

step3 = normalisasi(step2, kamus_normalisasi)
print(f'3. NORMALISASI       : {step3}')

step4 = tokenizing(step3)
print(f'4. TOKENIZING        : {step4}')

step5 = hapus_stopwords(step4)
print(f'5. STOPWORD REMOVAL  : {step5}')

step6 = stem(step5)
print(f'6. STEMMING          : {step6}')

hasil_akhir = ' '.join(step6)
print(f'\n   HASIL AKHIR       : {hasil_akhir}')
print('=' * 70)

## 7. Tabel Perbandingan Semua Tahap

In [ ]:
# Proses beberapa contoh ulasan tahap demi tahap
contoh = [
    'Apk nya bgs BGT tp lemot sma sering crash',
    'Gak bsa login udh update tp msh error',
    'Mantap bgt keren poll buat bljr bahasa',
    'Jelek parah gbs dipake hrs uninstall',
    'Makasih dev udh bikin app yg bgus'
]

data_tabel = []
for teks in contoh:
    s1 = case_folding(teks)
    s2 = cleaning(s1)
    s3 = normalisasi(s2, kamus_normalisasi)
    s4 = tokenizing(s3)
    s5 = hapus_stopwords(s4)
    s6 = stem(s5)
    
    data_tabel.append({
        'Teks Asli': teks,
        'Case Folding': s1,
        'Cleaning': s2,
        'Normalisasi': s3,
        'Tokenizing': str(s4),
        'Stopword Removal': str(s5),
        'Stemming': str(s6),
        'Hasil Akhir': ' '.join(s6)
    })

df_tahap = pd.DataFrame(data_tabel)
df_tahap.index = range(1, len(df_tahap) + 1)
df_tahap.index.name = 'No'

# Tampilkan per kolom agar lebih jelas
for col in df_tahap.columns:
    print(f'\n=== {col} ===')
    for i, val in enumerate(df_tahap[col], 1):
        print(f'  [{i}] {val}')

## Kesimpulan

Normalisasi kata merupakan tahap penting dalam preprocessing teks bahasa Indonesia, khususnya untuk ulasan dari Google Play Store yang banyak menggunakan bahasa tidak baku (slang, singkatan, bahasa gaul).

**Dalam aplikasi DuoSentimen:**
- Kamus normalisasi berisi **239 pasangan kata** slang → baku
- Mencakup kategori: singkatan umum, bahasa gaul, istilah teknologi
- Posisi dalam pipeline: setelah *Cleaning*, sebelum *Tokenizing*

---
**By Ahmad Saifulla** | DuoSentimen Project